# 1. Libraries

In [4]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler

### The below path should be where your data is saved (e.g. my data is savd in C:\Users\alexm\Documents\GitHub\SML_group_project\data")

In [5]:
import os
os.chdir(r"C:\Users\alexm\Documents\GitHub\SML_group_project")
print("Working directory:", os.getcwd())

Working directory: C:\Users\alexm\Documents\GitHub\SML_group_project


# 2. Load Data

In [6]:
chargers = pd.read_csv("data/EV_charger_locations.csv")
ev_reg = pd.read_csv("data/EV_registration_activity.csv")
cities_pop = pd.read_csv("data/cities_counties_pop.csv")
income = pd.read_csv("data/median_income_by_county.csv")

print("chargers:", chargers.shape)
print("ev_reg:", ev_reg.shape)
print("cities_pop:", cities_pop.shape)
print("income:", income.shape)

chargers: (1928, 42)
ev_reg: (1738602, 33)
cities_pop: (244, 5)
income: (39, 16)


# 3. Data Preparation #1: Data Cleaning

## 3.1 Clean EV registration data

In [7]:
ev = ev_reg.copy()

# Standardize column names
ev.columns = (
    ev.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

In [8]:
# Convert numeric columns
numeric_cols = [
    'model_year',
    'electric_range',
    'odometer_reading',
    'sale_price',
    'transaction_year'
]

for col in numeric_cols:
    if col in ev.columns:
        ev[col] = pd.to_numeric(ev[col], errors='coerce')

# Convert date columns
date_cols = [
    'sale_date',
    'transaction_date'
]

for col in date_cols:
    if col in ev.columns:
        ev[col] = pd.to_datetime(ev[col], errors='coerce')

In [9]:
# Replace placeholder zeros with NaN for matrix completion later
ev['sale_price'] = ev['sale_price'].replace(0, np.nan)
ev['electric_range'] = ev['electric_range'].replace(0, np.nan)

In [10]:
# Standardize county, city, and state text
for col in ['county', 'city', 'state']:
    if col in ev.columns:
        ev[col] = (
            ev[col]
            .replace(['nan', 'NaN', 'NAN', 'None', ''], np.nan)
            .astype(str)
            .str.strip()
            .str.upper()
        )

# Convert text "NAN" back to real missing values after uppercasing
ev['county'] = ev['county'].replace('NAN', np.nan)

# Create vehicle age
ev['vehicle_age'] = 2026 - ev['model_year']

# Drop EV rows with missing county because they cannot be used in county-level aggregation
ev = ev.dropna(subset=['county'])

ev.shape

(1738478, 34)

In [11]:
# Keep Washington records only
ev = ev[ev['state'] == 'WA'].copy()

# Drop rows without county
ev['county'] = ev['county'].replace(['NAN', 'nan', 'None', ''], np.nan)
ev = ev.dropna(subset=['county'])

## 3.2 Matrix completion note for `electric_range`

Placeholder zero values in `electric_range` were converted to missing values above. Matrix completion is applied later at the county level after aggregation. This avoids over smoothing millions of vehicle level rows.

## 3.3 Clean charger location data

In [12]:
ch = chargers.copy()

# Standardize column names
ch.columns = (
    ch.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

In [13]:
# Convert date columns
date_cols = [
    'date_last_',
    'date_las_1',
    'open_date',
    'availabili'
]

for col in date_cols:
    if col in ch.columns:
        ch[col] = pd.to_datetime(ch[col], errors='coerce')

# Standardize city and state text
for col in ['city', 'state']:
    if col in ch.columns:
        ch[col] = (
            ch[col]
            .astype(str)
            .str.strip()
            .str.upper()
        )

In [14]:
# Remove irrelevant text/ID fields for county-level analysis
drop_cols = [
    'x', 'y', 'fid', 'objectid', 'station_na', 'station_id',
    'address', 'directions', 'phone_numb',
    'ev_netwo_2', 'access_det', 'ev_pricing'
]

ch = ch.drop(columns=drop_cols, errors='ignore')

# Confirm duplicate and charger-count status
print("Duplicate charger rows:", ch.duplicated().sum())

charger_count_cols = ['ev_level1_', 'ev_level2_', 'ev_dc_fast', 'total_leve']
ch[charger_count_cols].isna().sum()

Duplicate charger rows: 0


ev_level1_    0
ev_level2_    0
ev_dc_fast    0
total_leve    0
dtype: int64

## 3.4 Clean city/county lookup data

In [15]:
cp = cities_pop.copy()

# Standardize column names
cp.columns = (
    cp.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

# Standardize city and county text
cp['city'] = (
    cp['city']
    .astype(str)
    .str.strip()
    .str.upper()
)

cp['county'] = (
    cp['county']
    .astype(str)
    .str.strip()
    .str.upper()
)

cp.head()

,city,population_2025,county,class,form_of_government
0,ABERDEEN,17200,GRAYS HARBOR,First,Mayor-Council
1,AIRWAY HEIGHTS,12120,SPOKANE,Code,Council-Manager
2,ALBION,545,WHITMAN,Town,Mayor-Council
3,ALGONA,3335,KING,Code,Mayor-Council
4,ALMIRA,315,LINCOLN,Town,Mayor-Council


## 3.5 Clean income data

In [16]:
income_2025 = income[['County', '2025']].copy()

income_2025.columns = ['county', 'median_income_2025']

income_2025['county'] = (
    income_2025['county']
    .str.upper()
    .str.strip()
)

income_2025['median_income_2025'] = pd.to_numeric(
    income_2025['median_income_2025'],
    errors='coerce'
)

income_2025.head()

,county,median_income_2025
0,ADAMS,69332.46
1,ASOTIN,75647.31
2,BENTON,95571.93
3,CHELAN,84324.53
4,CLALLAM,78138.51


# 4. Data Preparation #2: Add County to Charger Data

In [17]:
# Initial city to county merge
city_county = cp[['city', 'county']].drop_duplicates()

ch = ch.merge(
    city_county,
    on='city',
    how='left'
)

print("Missing charger county mappings after city merge:", ch['county'].isna().sum())

Missing charger county mappings after city merge: 235


In [18]:
# Manual fixes for charger cities not matched by the city/county lookup
manual_county_map = {
    'VANCOUVER': 'CLARK',
    'WALLA WALLA': 'WALLA WALLA',
    'TUKWILA': 'KING',
    'YAKIMA': 'YAKIMA',
    'WOODINVILLE': 'KING',
    'WENATCHEE': 'CHELAN',
    'WASHOUGAL': 'CLARK',
    'SILVERDALE': 'KITSAP',
    'TUMWATER': 'THURSTON',
    'MT. VERNON': 'SKAGIT',
    'MT VERNON': 'SKAGIT',
    'EASTSOUND': 'SAN JUAN',
    'WAPATO': 'YAKIMA',
    'KINGSTON': 'KITSAP',
    'WINTHROP': 'OKANOGAN',
    'MANSON': 'CHELAN',
    'NEAH BAY': 'CLALLAM',
    'WHITE SALMON': 'KLICKITAT',
    'ZILLAH': 'YAKIMA',
    'SNOQUALMIE PASS': 'KING',
    'LOPEZ': 'SAN JUAN',
    'LOPEZ ISLAND': 'SAN JUAN',
    'ASHFORD': 'PIERCE',
    'ROCKPORT': 'SKAGIT',
    'SUQUAMISH': 'KITSAP',
    'AURORA': 'KING',
    'MOCLIPS': 'GRAYS HARBOR',
    'BELFAIR': 'MASON',
    'TOPPENISH': 'YAKIMA',
    'ELBE': 'PIERCE',
    'SALKUM': 'LEWIS',
    'QUINAULT': 'GRAYS HARBOR',
    'CAMP MURRAY': 'PIERCE',
    'TULALIP': 'SNOHOMISH',
    'CAMANO': 'ISLAND',
    'WEST SEATTLE': 'KING',
    'VASHON': 'KING',
    'MEAD': 'SPOKANE',
    'WATERVILLE': 'DOUGLAS',
    'ROCHE HARBOR': 'SAN JUAN',
    'GRAYLAND': 'GRAYS HARBOR',
    'WAITSBURG': 'WALLA WALLA',
    'YELM': 'THURSTON',
    'MAZAMA': 'OKANOGAN',
    'COUGAR': 'COWLITZ',
    'CUSTER': 'WHATCOM',
    'TWISP': 'OKANOGAN',
    'TONASKET': 'OKANOGAN',
    'OLGA': 'SAN JUAN',
    'COPALIS CROSSING': 'GRAYS HARBOR',
    'ROCHESTER': 'THURSTON',
    'CARSON': 'SKAMANIA',
    'FREELAND': 'ISLAND',
    'SEAVIEW': 'PACIFIC',
    'UNION': 'MASON',
    'WESTPORT': 'GRAYS HARBOR',
    'FALL CITY': 'KING',
    'RICHLAND': 'BENTON',
    'RICHALND': 'BENTON',
    'BURIEN': 'KING',
    'LBURIEN': 'KING',
    'LYNNWOOD': 'SNOHOMISH',
    'LYNWOOD': 'SNOHOMISH',
    'SEA TAC': 'KING',
    'QUIL CEDA VILLAGE': 'SNOHOMISH',
    'PACIFIC BEACH': 'GRAYS HARBOR',
    'VISTA DR': 'SPOKANE',
    'ST': 'PIERCE'
}

ch['county'] = ch['county'].fillna(ch['city'].map(manual_county_map))

# Resolve ambiguous split-county city labels in charger data
ch.loc[ch['city'] == 'BOTHELL', 'county'] = 'KING'
ch.loc[ch['city'] == 'AUBURN', 'county'] = 'KING'

print("Missing charger county mappings after manual fixes:", ch['county'].isna().sum())

Missing charger county mappings after manual fixes: 0


# 5. Data Preparation #3: County-Level Aggregation

## 5.1 Aggregate EV adoption features by county

In [19]:
# Aggregate EV adoption features by county
ev_county = (
    ev.groupby('county')
    .agg(
        total_EV_count=('county', 'size'),
        BEV_count=(
            'clean_alternative_fuel_vehicle_type',
            lambda x: (x == 'Battery Electric Vehicle (BEV)').sum()
        ),
        PHEV_count=(
            'clean_alternative_fuel_vehicle_type',
            lambda x: (x == 'Plug-in Hybrid Electric Vehicle (PHEV)').sum()
        ),
        avg_model_year=('model_year', 'mean'),
        avg_electric_range=('electric_range', 'mean'),
        avg_vehicle_age=('vehicle_age', 'mean')
    )
    .reset_index()
)

# Derived EV features
ev_county['percent_BEV'] = ev_county['BEV_count'] / ev_county['total_EV_count']
ev_county['percent_PHEV'] = ev_county['PHEV_count'] / ev_county['total_EV_count']

# Avoid divide by zero
ev_county['BEV_to_PHEV_ratio'] = (
    ev_county['BEV_count'] /
    ev_county['PHEV_count'].replace(0, np.nan)
)

ev_county.head()

,county,total_EV_count,BEV_count,PHEV_count,avg_model_year,avg_electric_range,avg_vehicle_age,percent_BEV,percent_PHEV,BEV_to_PHEV_ratio
0,ADAMS,620,435,185,2020.061290,111.452722,5.938710,0.701613,0.298387,2.351351
1,ASOTIN,710,452,258,2019.712676,95.222698,6.287324,0.636620,0.363380,1.751938
2,BENTON,22639,15493,7146,2019.935200,108.205155,6.064800,0.684350,0.315650,2.168066
3,CHELAN,10014,7883,2131,2020.235171,125.745458,5.764829,0.787198,0.212802,3.699202
4,CLALLAM,10602,7089,3513,2019.356725,98.694111,6.643275,0.668647,0.331353,2.017933


## 5.2 County-level matrix completion for `avg_electric_range`

This performs the matrix-completion step, but at the county level rather than the individual vehicle level. The observed county averages are kept fixed, and only missing county level `avg_electric_range` values are completed. If no counties have missing averages, the column is left unchanged, which is what happens in this final version.

In [20]:
# Matrix completion at county level for avg_electric_range
range_impute_cols = [
    'total_EV_count',
    'BEV_count',
    'PHEV_count',
    'percent_BEV',
    'percent_PHEV',
    'avg_model_year',
    'avg_vehicle_age',
    'avg_electric_range',
    'BEV_to_PHEV_ratio'
]

# Keep a copy of the observed county averages for transparency
ev_county['avg_electric_range_observed'] = ev_county['avg_electric_range']

missing_avg_range = ev_county['avg_electric_range'].isna()
print('Missing county avg_electric_range before matrix completion:', missing_avg_range.sum())

if missing_avg_range.sum() > 0:
    range_matrix = ev_county[range_impute_cols].replace([np.inf, -np.inf], np.nan).copy()

    scaler = StandardScaler()
    range_scaled = scaler.fit_transform(range_matrix)

    imputer = IterativeImputer(
        max_iter=20,
        random_state=42,
        initial_strategy='median'
    )

    range_completed_scaled = imputer.fit_transform(range_scaled)

    range_completed = pd.DataFrame(
        scaler.inverse_transform(range_completed_scaled),
        columns=range_impute_cols,
        index=ev_county.index
    )

    ev_county.loc[missing_avg_range, 'avg_electric_range'] = (
        range_completed.loc[missing_avg_range, 'avg_electric_range']
    )

# Electric range cannot be negative
ev_county['avg_electric_range'] = ev_county['avg_electric_range'].clip(lower=0)

print('Missing county avg_electric_range after matrix completion:', ev_county['avg_electric_range'].isna().sum())

ev_county[['county', 'avg_electric_range_observed', 'avg_electric_range']].head()

Missing county avg_electric_range before matrix completion: 0
Missing county avg_electric_range after matrix completion: 0


,county,avg_electric_range_observed,avg_electric_range
0,ADAMS,111.452722,111.452722
1,ASOTIN,95.222698,95.222698
2,BENTON,108.205155,108.205155
3,CHELAN,125.745458,125.745458
4,CLALLAM,98.694111,98.694111


Electric range contained missing values after placeholder zeros were converted to NaN. Matrix completion was tested, but around 44% of vehicle records required imputation. At the vehicle level, this produced overly similar electric range values and reduced natural variation in the data suggesting over-smoothing.

Because county level aggregation used the mean of observed values (mean() ignores NaNs), all counties still retained valid average electric range estimates and no county level missing values remained.

Since the goal of this project is county level PCA and clustering, preserving observed county level patterns was preferred over imputing a large number of vehicle level values. County averages were therefore calculated from observed data, with matrix completion evaluated but not actually needed in the final dataset.

## 5.3 Merge county-level income

In [21]:
ev_county = ev_county.merge(
    income_2025,
    on='county',
    how='left'
)

print("Missing income values:", ev_county['median_income_2025'].isna().sum())
ev_county.head()

Missing income values: 0


,county,total_EV_count,BEV_count,PHEV_count,avg_model_year,avg_electric_range,avg_vehicle_age,percent_BEV,percent_PHEV,BEV_to_PHEV_ratio,avg_electric_range_observed,median_income_2025
0,ADAMS,620,435,185,2020.061290,111.452722,5.938710,0.701613,0.298387,2.351351,111.452722,69332.46
1,ASOTIN,710,452,258,2019.712676,95.222698,6.287324,0.636620,0.363380,1.751938,95.222698,75647.31
2,BENTON,22639,15493,7146,2019.935200,108.205155,6.064800,0.684350,0.315650,2.168066,108.205155,95571.93
3,CHELAN,10014,7883,2131,2020.235171,125.745458,5.764829,0.787198,0.212802,3.699202,125.745458,84324.53
4,CLALLAM,10602,7089,3513,2019.356725,98.694111,6.643275,0.668647,0.331353,2.017933,98.694111,78138.51


## 5.4 Population feature note

Attempted to make `EV_per_1000_people`, but the `cities_counties_pop` file appears to contain city level populations rather than complete county populations. Summing those city values produced impossible EV-per-1000 values, so `Population_2025` and `EV_per_1000_people` were excluded from the final cleaned dataset. `Population_density` was also not created because we don't have land area data. We would need another dataset for this.

## 5.5 Aggregate charger features by county

In [22]:
charger_county = (
    ch.groupby('county')
    .agg(
        total_chargers=('total_leve', 'sum'),
        level1_chargers=('ev_level1_', 'sum'),
        level2_chargers=('ev_level2_', 'sum'),
        dc_fast_chargers=('ev_dc_fast', 'sum')
    )
    .reset_index()
)

charger_county['fast_charger_share'] = (
    charger_county['dc_fast_chargers'] /
    charger_county['total_chargers']
)

charger_county.head()

,county,total_chargers,level1_chargers,level2_chargers,dc_fast_chargers,fast_charger_share
0,ADAMS,9,0,0,9,1.000000
1,ASOTIN,2,0,2,0,0.000000
2,BENTON,45,0,30,15,0.333333
3,CHELAN,80,12,53,27,0.337500
4,CLALLAM,68,0,45,23,0.338235


# 6. Final County-Level Dataset

In [23]:
# Keep the final feature set concise
ev_county_final = ev_county.drop(columns=['avg_electric_range_observed'], errors='ignore')

county_data = ev_county_final.merge(
    charger_county,
    on='county',
    how='left'
)

# Counties with no charger records should have charger counts of 0
charger_cols = [
    'total_chargers',
    'level1_chargers',
    'level2_chargers',
    'dc_fast_chargers',
    'fast_charger_share'
]

county_data[charger_cols] = county_data[charger_cols].fillna(0)

county_data['chargers_per_1000_EV'] = (
    county_data['total_chargers'] /
    county_data['total_EV_count']
) * 1000

county_data.head()

,county,total_EV_count,BEV_count,PHEV_count,avg_model_year,avg_electric_range,avg_vehicle_age,percent_BEV,percent_PHEV,BEV_to_PHEV_ratio,median_income_2025,total_chargers,level1_chargers,level2_chargers,dc_fast_chargers,fast_charger_share,chargers_per_1000_EV
0,ADAMS,620,435,185,2020.061290,111.452722,5.938710,0.701613,0.298387,2.351351,69332.46,9.0,0.0,0.0,9.0,1.000000,14.516129
1,ASOTIN,710,452,258,2019.712676,95.222698,6.287324,0.636620,0.363380,1.751938,75647.31,2.0,0.0,2.0,0.0,0.000000,2.816901
2,BENTON,22639,15493,7146,2019.935200,108.205155,6.064800,0.684350,0.315650,2.168066,95571.93,45.0,0.0,30.0,15.0,0.333333,1.987720
3,CHELAN,10014,7883,2131,2020.235171,125.745458,5.764829,0.787198,0.212802,3.699202,84324.53,80.0,12.0,53.0,27.0,0.337500,7.988816
4,CLALLAM,10602,7089,3513,2019.356725,98.694111,6.643275,0.668647,0.331353,2.017933,78138.51,68.0,0.0,45.0,23.0,0.338235,6.413884


# 7. Final Validation

In [24]:
print("Final dataset shape:", county_data.shape)
print("Missing values:")
print(county_data.isna().sum())

county_data.describe()

Final dataset shape: (39, 17)
Missing values:
county                  0
total_EV_count          0
BEV_count               0
PHEV_count              0
avg_model_year          0
avg_electric_range      0
avg_vehicle_age         0
percent_BEV             0
percent_PHEV            0
BEV_to_PHEV_ratio       0
median_income_2025      0
total_chargers          0
level1_chargers         0
level2_chargers         0
dc_fast_chargers        0
fast_charger_share      0
chargers_per_1000_EV    0
dtype: int64


,total_EV_count,BEV_count,PHEV_count,avg_model_year,avg_electric_range,avg_vehicle_age,percent_BEV,percent_PHEV,BEV_to_PHEV_ratio,median_income_2025,total_chargers,level1_chargers,level2_chargers,dc_fast_chargers,fast_charger_share,chargers_per_1000_EV
count,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000,39.000000
mean,44397.487179,34171.076923,10225.307692,2019.882930,106.786601,6.117070,0.687231,0.312563,2.395558,83781.304103,119.205128,4.743590,96.230769,22.974359,0.239174,5.937223
std,145873.829414,116333.511156,29630.964809,0.514577,14.701700,0.514577,0.087216,0.087304,0.789489,15515.654597,392.752043,25.927752,344.472248,50.267959,0.285324,8.548014
min,34.000000,11.000000,23.000000,2018.000000,62.566667,5.322807,0.323529,0.193865,0.478261,60663.480000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2100.000000,1273.500000,809.000000,2019.706917,99.941830,5.837962,0.660433,0.266212,1.945706,72026.225000,9.500000,0.000000,8.000000,0.000000,0.000000,2.261317
50%,6469.000000,4408.000000,2038.000000,2019.995672,105.823985,6.004328,0.700193,0.299807,2.335477,81477.360000,26.000000,0.000000,15.000000,9.000000,0.142857,3.400835
75%,19038.500000,13667.500000,5371.000000,2020.162038,113.543127,6.293083,0.733781,0.339567,2.756440,92797.145000,77.000000,0.000000,54.000000,29.000000,0.335417,6.462633
max,894695.000000,713944.000000,180748.000000,2020.677193,141.765027,8.000000,0.806135,0.676471,4.158220,126124.370000,2465.000000,162.000000,2162.000000,303.000000,1.000000,51.948052


# 8. Save Cleaned Data

In [25]:
county_data.to_csv("data/clean_county_data.csv", index=False)
ch.to_csv("data/clean_chargers.csv", index=False)
charger_county.to_csv("data/charger_county.csv", index=False)
ev_county.to_csv("data/ev_county.csv", index=False)

print("Saved cleaned datasets.")

Saved cleaned datasets.
